<a href="https://colab.research.google.com/github/Kr-Suryansh/BIS_LAB/blob/main/ACO%20(Electrical%20Grid).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import numpy as np

# 1. Define the Problem: Set of cities with (x, y) coordinates
cities = np.array([
    [0.0, 0.0],
    [1.0, 3.0],
    [4.0, 3.0],
    [6.0, 1.0],
    [3.0, 0.0]
])

num_cities = len(cities)

# Calculate distance matrix between all pairs of cities
dist_matrix = np.zeros((num_cities, num_cities))
for i in range(num_cities):
    for j in range(num_cities):
        dist_matrix[i, j] = np.linalg.norm(cities[i] - cities[j])
        if i == j:
            dist_matrix[i, j] = np.inf  # Prevent self-loops

# 2. Initialize Parameters
num_ants = 5
num_iterations = 100
alpha = 1.0       # Importance of pheromone trail
beta = 2.0        # Importance of heuristic information (inverse distance)
rho = 0.5         # Pheromone evaporation rate
initial_pheromone = 1.0
pheromone = np.ones((num_cities, num_cities)) * initial_pheromone

# Heuristic information: eta = 1 / distance
eta = np.zeros((num_cities, num_cities))
for i in range(num_cities):
    for j in range(num_cities):
        if i != j:
            eta[i, j] = 1.0 / dist_matrix[i, j]

best_tour = None
best_distance = float('inf')

# 5. Iterate
for iteration in range(num_iterations):
    all_ant_tours = []
    all_ant_distances = []

    # 3. Construct Solutions for each ant
    for ant in range(num_ants):
        visited = []
        # Start at a random city
        current_city = np.random.randint(num_cities)
        visited.append(current_city)

        while len(visited) < num_cities:
            unvisited = [c for c in range(num_cities) if c not in visited]

            # Calculate transition probabilities for unvisited cities
            probabilities = []
            for next_city in unvisited:
                tau = pheromone[current_city, next_city] ** alpha
                eta_val = eta[current_city, next_city] ** beta
                probabilities.append(tau * eta_val)

            probabilities = np.array(probabilities)
            probabilities /= probabilities.sum()

            # Choose next city probabilistically
            next_city = np.random.choice(unvisited, p=probabilities)
            visited.append(next_city)
            current_city = next_city

        # Complete the tour by returning to the starting city
        tour = visited
        distance = sum(dist_matrix[tour[i], tour[(i + 1) % num_cities]] for i in range(num_cities))

        all_ant_tours.append(tour)
        all_ant_distances.append(distance)

        # Track the global best solution
        if distance < best_distance:
            best_distance = distance
            best_tour = tour

    # 4. Update Pheromones
    # Evaporate pheromones
    pheromone *= (1.0 - rho)

    # Deposit new pheromones based on ant tour quality
    for tour, distance in zip(all_ant_tours, all_ant_distances):
        for i in range(num_cities):
            u = tour[i]
            v = tour[(i + 1) % num_cities]
            pheromone[u, v] += 1.0 / distance
            pheromone[v, u] += 1.0 / distance  # Symmetrical TSP

# 6. Output the Best Solution
formatted_tour = [int(node) for node in best_tour]
print(f"Best Tour Found: {formatted_tour}")
print(f"Best Distance: {best_distance}")

Best Tour Found: [4, 3, 2, 1, 0]
Best Distance: 15.15298244508295


Power Distribution Optimization – Minimizing power loss in electrical distribution networks.

In [3]:
import numpy as np
from collections import deque

# ==========================================
# 1. DEFINE THE DISTRIBUTION NETWORK DATA
# ==========================================
# Substation is Bus 0 (Slack Bus, V = 1.0 p.u.)
num_buses = 12

# Active (P_kW) and Reactive (Q_kVAr) loads at buses 1 to 11
# Index 0 is substation (0 load)
P_load = np.array([0, 60, 40, 55, 30, 70, 45, 80, 50, 65, 35, 90], dtype=float)
Q_load = np.array([0, 30, 20, 25, 15, 35, 20, 40, 25, 30, 15, 45], dtype=float)

# Base values for Per-Unit (p.u.) conversion
S_base = 1000.0  # 1 MVA = 1000 kVA
V_base = 12.66   # 12.66 kV
Z_base = (V_base ** 2) / (S_base / 1000.0)  # Ohms

P_pu = P_load / S_base
Q_pu = Q_load / S_base

# Candidate branches: (from_bus, to_bus, R_ohms, X_ohms)
# Branches 0-10 are normally closed sectionalizing switches
# Branches 11-13 are normally open tie-switches creating loops
branches = [
    (0, 1, 0.0922, 0.0470),
    (1, 2, 0.4930, 0.2511),
    (2, 3, 0.3660, 0.1864),
    (3, 4, 0.3811, 0.1941),
    (1, 5, 0.8190, 0.7070),
    (5, 6, 0.1872, 0.6188),
    (6, 7, 0.7114, 0.2351),
    (2, 8, 1.0300, 0.7400),
    (8, 9, 1.0440, 0.7400),
    (9, 10, 0.1966, 0.0650),
    (10, 11, 0.3744, 0.1238),
    # Tie-lines (normally open)
    (4, 7, 0.5000, 0.5000),
    (7, 11, 0.4500, 0.4500),
    (3, 9, 0.6000, 0.6000)
]

num_branches = len(branches)
R_pu = np.array([b[2] / Z_base for b in branches])
X_pu = np.array([b[3] / Z_base for b in branches])

# Heuristic info: inverse of branch resistance (favor low-resistance lines)
eta = 1.0 / R_pu


# ==========================================
# 2. RADIAL TOPOLOGY & LOAD FLOW EVALUATOR
# ==========================================
def evaluate_network(active_branch_indices):
    """
    Runs Backward/Forward Sweep Load Flow on a radial tree.
    Returns (total_power_loss_kW, min_voltage_pu, is_valid)
    """
    # Build adjacency list for the active tree
    adj = {i: [] for i in range(num_buses)}
    for b_idx in active_branch_indices:
        u, v, _, _ = branches[b_idx]
        adj[u].append((v, b_idx))
        adj[v].append((u, b_idx))

    # BFS from Substation (Bus 0) to establish parent-child radial hierarchy
    parent = {0: None}
    parent_branch = {0: None}
    order = []
    queue = deque([0])

    while queue:
        curr = queue.popleft()
        order.append(curr)
        for nxt, b_idx in adj[curr]:
            if nxt not in parent:
                parent[nxt] = curr
                parent_branch[nxt] = b_idx
                queue.append(nxt)

    # Check if all buses are energized (connected spanning tree)
    if len(order) != num_buses:
        return float('inf'), 0.0, False

    # Backward/Forward Sweep Iterative Power Flow
    V = np.ones(num_buses, dtype=complex)
    for _ in range(15):  # Converges in < 10 iterations for radial systems
        # Backward Sweep: Accumulate branch currents from leaf nodes to root
        I_bus = np.conj((P_pu + 1j * Q_pu) / V)
        I_branch = np.zeros(num_branches, dtype=complex)

        for bus in reversed(order[1:]):
            b_idx = parent_branch[bus]
            I_branch[b_idx] = I_bus[bus]
            p = parent[bus]
            I_bus[p] += I_branch[b_idx]

        # Forward Sweep: Update bus voltages from root to leaves
        for bus in order[1:]:
            p = parent[bus]
            b_idx = parent_branch[bus]
            Z = R_pu[b_idx] + 1j * X_pu[b_idx]
            V[bus] = V[p] - I_branch[b_idx] * Z

    # Calculate total active power loss (I^2 * R) in kW
    total_loss_pu = sum((np.abs(I_branch[b]) ** 2) * R_pu[b] for b in active_branch_indices)
    total_loss_kw = total_loss_pu * S_base
    min_v = np.min(np.abs(V))

    # Apply penalty if voltage drops below 0.95 p.u.
    if min_v < 0.95:
        total_loss_kw += 1000.0 * (0.95 - min_v)

    return total_loss_kw, min_v, True


# ==========================================
# 3. ANT COLONY OPTIMIZATION (ACO)
# ==========================================
def run_aco_dnr(num_ants=15, num_iterations=60, alpha=1.0, beta=2.0, rho=0.3):
    pheromone = np.ones(num_branches) * 1.0
    best_closed_branches = None
    best_loss = float('inf')
    best_min_v = 0.0

    # Baseline loss (initial configuration: branches 0 to 10 closed)
    base_loss, base_v, _ = evaluate_network(list(range(11)))
    print(f"Initial Network Loss: {base_loss:.2f} kW | Min Voltage: {base_v:.4f} p.u.")

    for iteration in range(num_iterations):
        ant_solutions = []
        ant_losses = []

        for _ in range(num_ants):
            # Construct a spanning tree (Prim-like randomized probabilistic growth)
            visited_buses = {0}
            closed_branches = []

            while len(visited_buses) < num_buses:
                # Find frontier branches connecting visited to unvisited buses
                frontier = []
                for b_idx, (u, v, _, _) in enumerate(branches):
                    if (u in visited_buses) ^ (v in visited_buses):
                        frontier.append(b_idx)

                # Calculate selection probabilities using pheromone and inverse resistance
                probs = np.array([(pheromone[b] ** alpha) * (eta[b] ** beta) for b in frontier])
                probs /= probs.sum()

                # Probabilistic selection
                chosen_branch = np.random.choice(frontier, p=probs)
                closed_branches.append(chosen_branch)

                u, v, _, _ = branches[chosen_branch]
                visited_buses.add(u)
                visited_buses.add(v)

            # Evaluate candidate radial network
            loss, min_v, valid = evaluate_network(closed_branches)
            if valid:
                ant_solutions.append(closed_branches)
                ant_losses.append(loss)

                if loss < best_loss:
                    best_loss = loss
                    best_closed_branches = sorted(closed_branches)
                    best_min_v = min_v

        # Evaporate pheromones
        pheromone *= (1.0 - rho)

        # Deposit pheromones proportional to inverse power loss
        for closed_branches, loss in zip(ant_solutions, ant_losses):
            deposit = 10.0 / loss
            for b_idx in closed_branches:
                pheromone[b_idx] += deposit

    open_switches = [b for b in range(num_branches) if b not in best_closed_branches]
    return best_closed_branches, open_switches, best_loss, best_min_v


# Execute Optimization
best_closed, best_open, min_loss, min_voltage = run_aco_dnr()

print("\n--- ACO Optimization Results ---")
print(f"Optimal Open Tie-Switches (Branch Indices): {best_open}")
print(f"Minimized Power Loss: {min_loss:.2f} kW")
print(f"Minimum Bus Voltage:  {min_voltage:.4f} p.u.")


Initial Network Loss: 1.90 kW | Min Voltage: 0.9940 p.u.

--- ACO Optimization Results ---
Optimal Open Tie-Switches (Branch Indices): [8, 10, 11]
Minimized Power Loss: 1.50 kW
Minimum Bus Voltage:  0.9955 p.u.


In [10]:
import numpy as np
from collections import deque

# ==========================================
# 1. DEFINE THE DISTRIBUTION NETWORK DATA
# ==========================================
num_buses = 12

P_load = np.array([0, 60, 40, 55, 30, 70, 45, 80, 50, 65, 35, 90], dtype=float)
Q_load = np.array([0, 30, 20, 25, 15, 35, 20, 40, 25, 30, 15, 45], dtype=float)

S_base = 1000.0  # 1 MVA = 1000 kVA
V_base = 12.66   # 12.66 kV
Z_base = (V_base ** 2) / (S_base / 1000.0)  # Ohms

P_pu = P_load / S_base
Q_pu = Q_load / S_base

branches = [
    (0, 1, 0.0922, 0.0470),
    (1, 2, 0.4930, 0.2511),
    (2, 3, 0.3660, 0.1864),
    (3, 4, 0.3811, 0.1941),
    (1, 5, 0.8190, 0.7070),
    (5, 6, 0.1872, 0.6188),
    (6, 7, 0.7114, 0.2351),
    (2, 8, 1.0300, 0.7400),
    (8, 9, 1.0440, 0.7400),
    (9, 10, 0.1966, 0.0650),
    (10, 11, 0.3744, 0.1238),
    # Tie-lines (normally open)
    (4, 7, 0.5000, 0.5000),
    (7, 11, 0.4500, 0.4500),
    (3, 9, 0.6000, 0.6000)
]

num_branches = len(branches)
R_pu = np.array([b[2] / Z_base for b in branches])
X_pu = np.array([b[3] / Z_base for b in branches])

eta = 1.0 / R_pu


# ==========================================
# 2. RADIAL TOPOLOGY & LOAD FLOW EVALUATOR
# ==========================================
def evaluate_network(active_branch_indices):
    adj = {i: [] for i in range(num_buses)}
    for b_idx in active_branch_indices:
        u, v, _, _ = branches[b_idx]
        adj[u].append((v, b_idx))
        adj[v].append((u, b_idx))

    parent = {0: None}
    parent_branch = {0: None}
    order = []
    queue = deque([0])

    while queue:
        curr = queue.popleft()
        order.append(curr)
        for nxt, b_idx in adj[curr]:
            if nxt not in parent:
                parent[nxt] = curr
                parent_branch[nxt] = b_idx
                queue.append(nxt)

    if len(order) != num_buses:
        return float('inf'), 0.0, False

    V = np.ones(num_buses, dtype=complex)
    for _ in range(15):
        I_bus = np.conj((P_pu + 1j * Q_pu) / V)
        I_branch = np.zeros(num_branches, dtype=complex)

        for bus in reversed(order[1:]):
            b_idx = parent_branch[bus]
            I_branch[b_idx] = I_bus[bus]
            p = parent[bus]
            I_bus[p] += I_branch[b_idx]

        for bus in order[1:]:
            p = parent[bus]
            b_idx = parent_branch[bus]
            Z = R_pu[b_idx] + 1j * X_pu[b_idx]
            V[bus] = V[p] - I_branch[b_idx] * Z

    total_loss_pu = sum((np.abs(I_branch[b]) ** 2) * R_pu[b] for b in active_branch_indices)
    total_loss_kw = total_loss_pu * S_base
    min_v = np.min(np.abs(V))

    if min_v < 0.95:
        total_loss_kw += 1000.0 * (0.95 - min_v)

    return total_loss_kw, min_v, True


# ==========================================
# 3. ANT COLONY OPTIMIZATION (ACO)
# ==========================================
def run_aco_dnr(num_ants=15, num_iterations=60, alpha=1.0, beta=2.0, rho=0.3):
    pheromone = np.ones(num_branches) * 1.0
    best_closed_branches = None
    best_loss = float('inf')
    best_min_v = 0.0

    base_loss, base_v, _ = evaluate_network(list(range(11)))
    print(f"Initial Network Loss: {base_loss:.2f} kW | Min Voltage: {base_v:.4f} p.u.")

    for iteration in range(num_iterations):
        ant_solutions = []
        ant_losses = []

        for _ in range(num_ants):
            visited_buses = {0}
            closed_branches = []

            while len(visited_buses) < num_buses:
                frontier = []
                for b_idx, (u, v, _, _) in enumerate(branches):
                    if (u in visited_buses) ^ (v in visited_buses):
                        frontier.append(b_idx)

                probs = np.array([(pheromone[b] ** alpha) * (eta[b] ** beta) for b in frontier])
                probs /= probs.sum()

                chosen_branch = np.random.choice(frontier, p=probs)
                closed_branches.append(chosen_branch)

                u, v, _, _ = branches[chosen_branch]
                visited_buses.add(u)
                visited_buses.add(v)

            loss, min_v, valid = evaluate_network(closed_branches)
            if valid:
                ant_solutions.append(closed_branches)
                ant_losses.append(loss)

                if loss < best_loss:
                    best_loss = loss
                    best_closed_branches = sorted(closed_branches)
                    best_min_v = min_v

        pheromone *= (1.0 - rho)

        for closed_branches, loss in zip(ant_solutions, ant_losses):
            deposit = 10.0 / loss
            for b_idx in closed_branches:
                pheromone[b_idx] += deposit

    open_switches = [b for b in range(num_branches) if b not in best_closed_branches]
    return best_closed_branches, open_switches, best_loss, best_min_v


# ==========================================
# 4. OPTIMAL PATH DISPLAY HELPER
# ==========================================
def print_optimal_paths(closed_branches):
    adj = {i: [] for i in range(num_buses)}
    for b_idx in closed_branches:
        u, v, _, _ = branches[b_idx]
        adj[u].append(v)
        adj[v].append(u)

    paths = {0: [0]}
    queue = deque([0])
    while queue:
        curr = queue.popleft()
        for nxt in sorted(adj[curr]):
            if nxt not in paths:
                paths[nxt] = paths[curr] + [nxt]
                queue.append(nxt)

    print("\n--- Optimal Closed Switches (Active Connections) ---")
    for b_idx in closed_branches:
        u, v, r, _ = branches[b_idx]
        print(f"  Branch {b_idx:2d}: Bus {u:2d} <--> Bus {v:2d} (R = {r:.4f} Ohms)")

    print("\n--- Optimal Power Delivery Paths (From Substation Bus 0) ---")
    for bus in range(1, num_buses):
        route_str = " -> ".join(f"Bus {b}" for b in paths[bus])
        print(f"  To Bus {bus:2d}: {route_str}")


# Execute Optimization
best_closed, best_open, min_loss, min_voltage = run_aco_dnr()

print("\n--- ACO Optimization Results ---")
print(f"Optimal Open Tie-Switches (Branch Indices): {best_open}")
print(f"Minimized Power Loss: {min_loss:.2f} kW")
print(f"Minimum Bus Voltage:  {min_voltage:.4f} p.u.")

print_optimal_paths(best_closed)

Initial Network Loss: 1.90 kW | Min Voltage: 0.9940 p.u.

--- ACO Optimization Results ---
Optimal Open Tie-Switches (Branch Indices): [8, 10, 11]
Minimized Power Loss: 1.50 kW
Minimum Bus Voltage:  0.9955 p.u.

--- Optimal Closed Switches (Active Connections) ---
  Branch  0: Bus  0 <--> Bus  1 (R = 0.0922 Ohms)
  Branch  1: Bus  1 <--> Bus  2 (R = 0.4930 Ohms)
  Branch  2: Bus  2 <--> Bus  3 (R = 0.3660 Ohms)
  Branch  3: Bus  3 <--> Bus  4 (R = 0.3811 Ohms)
  Branch  4: Bus  1 <--> Bus  5 (R = 0.8190 Ohms)
  Branch  5: Bus  5 <--> Bus  6 (R = 0.1872 Ohms)
  Branch  6: Bus  6 <--> Bus  7 (R = 0.7114 Ohms)
  Branch  7: Bus  2 <--> Bus  8 (R = 1.0300 Ohms)
  Branch  9: Bus  9 <--> Bus 10 (R = 0.1966 Ohms)
  Branch 12: Bus  7 <--> Bus 11 (R = 0.4500 Ohms)
  Branch 13: Bus  3 <--> Bus  9 (R = 0.6000 Ohms)

--- Optimal Power Delivery Paths (From Substation Bus 0) ---
  To Bus  1: Bus 0 -> Bus 1
  To Bus  2: Bus 0 -> Bus 1 -> Bus 2
  To Bus  3: Bus 0 -> Bus 1 -> Bus 2 -> Bus 3
  To Bus  4: